# MLP Digit Addition

One shared MLP with two output heads:
- **ones head** — predicts (a+b) % 10
- **carry head** — predicts (a+b) // 10

Three models: A trains ones only, B trains both from scratch, C copies A and fine-tunes with the first layer frozen.

## Imports

In [95]:
import torch  # PyTorch for tensors and autograd
import time   # used to measure training duration for each model

## Creates the dataset

In [96]:
X       = torch.zeros(100, 20)               # input of 100 pairs, two 10-dim one-hot vectors (10 for a, 10 for b)
Y_ones  = torch.zeros(100, dtype=torch.long) # value of ones head
Y_carry = torch.zeros(100, dtype=torch.long) # value of carry head

for a in range(10):       # first digits
    for b in range(10):   # second digits
        i = a * 10 + b    # find the right row for the pair (a, b)
        X[i, a]      = 1.0            # set the a-th bit in the first 10 positions to encode digit a
        X[i, 10 + b] = 1.0            # set the b-th bit in the last 10 positions to encode digit b
        Y_ones[i]    = (a + b) % 10   # store the ones digit of the sum
        Y_carry[i]   = (a + b) // 10  # store the carry digit (0 or 1)

#print
print('Y_ones shape :', Y_ones.shape)     
print('Y_carry shape:', Y_carry.shape)    
print('Print dataset:')
for i in range(100): 
    print(f'  X={X[i].tolist()}  ones={Y_ones[i].item()}  carry={Y_carry[i].item()}')

Y_ones shape : torch.Size([100])
Y_carry shape: torch.Size([100])
Print dataset:
  X=[1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]  ones=0  carry=0
  X=[1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]  ones=1  carry=0
  X=[1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]  ones=2  carry=0
  X=[1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]  ones=3  carry=0
  X=[1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0]  ones=4  carry=0
  X=[1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0]  ones=5  carry=0
  X=[1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0]  ones=6  carry=0
  X=[1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 

## Architecture


input (20) -> hidden layer 1 (64) (frozen in Model C) -> hidden layer 2 (64) -> ones head (10 classes) and carry head (2 classes)


In [97]:
def make_weights():
    W1  = torch.randn(20, 64)  * 0.1  # first hidden layer weight matrix: 20 -> 64
    b1  = torch.zeros(64)             # bias for first hidden layer
    W2  = torch.randn(64, 64)  * 0.1  # second hidden layer weight matrix: 64 -> 64
    b2  = torch.zeros(64)             # bias for second hidden layer
    Wo  = torch.randn(64, 10)  * 0.1  # ones-head output weights: 64 -> 10 class logits
    bo  = torch.zeros(10)             # ones-head output bias
    Wc  = torch.randn(64, 2)   * 0.1  # carry-head output weights: 64 -> 2 class logits
    bc  = torch.zeros(2)              # carry-head output bias
    return W1, b1, W2, b2, Wo, bo, Wc, bc

def forward(X, W1, b1, W2, b2, Wo, bo, Wc, bc):
    h1          = (X  @ W1 + b1).clamp(min=0)   # linear transform then ReLU activation: first layer
    h2          = (h1 @ W2 + b2).clamp(min=0)   # linear transform then ReLU activation: second layer
    logits_ones  = h2 @ Wo + bo                 # linear projection to 10 logits for the ones digit
    logits_carry = h2 @ Wc + bc                 # linear projection to 2 logits for the carry digit
    return logits_ones, logits_carry

def cross_entropy(logits, targets):
    log_probs = logits - logits.exp().sum(dim=1, keepdim=True).log()  # log-softmax: subtract log of sum of exps for numerical stability
    return -log_probs[range(len(targets)), targets].mean()             # pick the log-prob of the correct class and negate the mean

def accuracy(logits, targets):
    return (logits.argmax(dim=1) == targets).sum().item()  # count how many predictions match the targets

lr = 0.01  # learning rate used by all three models
print('architecture and helpers defined')

architecture and helpers defined


## Model A — ones digit only

In [98]:
W1_a, b1_a, W2_a, b2_a, Wo_a, bo_a, Wc_a, bc_a = make_weights()  # new set of weights
params_a = [W1_a, b1_a, W2_a, b2_a, Wo_a, bo_a]                  # Wc_a and bc_a are excluded: carry head receives no gradient
for p in params_a:
    p.requires_grad_(True)  # enable gradient tracking for each included parameter

t0_a = time.time()  # record start time so we can measure total training duration

for epoch in range(1, 50001): 

    logits_ones, logits_carry = forward(X, W1_a, b1_a, W2_a, b2_a, Wo_a, bo_a, Wc_a, bc_a)  # forward pass

    loss = cross_entropy(logits_ones, Y_ones)  # ones-head loss; carry is masked

    loss.backward()  # backpropagate gradients through all parameters

    with torch.no_grad():          # disable autograd during the weight update step
        for p in params_a:
            p -= lr * p.grad       # nudge each weight
            p.grad.zero_()         # reset gradient to zero for the next iteration

    correct_ones = accuracy(logits_ones, Y_ones)  # count correctly predicted ones digits

    if epoch % 1000 == 0:
        print(f'epoch {epoch:5d}  loss={loss.item():.4f}  ones={correct_ones}/100')  #  progress update

    if correct_ones == 100:  # early stop once 100%
        print(f'Model A reached 100% ones accuracy at epoch {epoch}')
        break

time_a = time.time() - t0_a 
print(f'Model A training time: {time_a:.2f}s')

epoch  1000  loss=2.2868  ones=20/100
epoch  2000  loss=2.2710  ones=29/100
epoch  3000  loss=2.2511  ones=34/100
epoch  4000  loss=2.2237  ones=44/100
epoch  5000  loss=2.1839  ones=50/100
epoch  6000  loss=2.1195  ones=59/100
epoch  7000  loss=2.0068  ones=64/100
epoch  8000  loss=1.8006  ones=73/100
epoch  9000  loss=1.4405  ones=85/100
epoch 10000  loss=0.9475  ones=93/100
Model A reached 100% ones accuracy at epoch 10609
Model A training time: 10.22s


## Verify Model A

In [99]:
with torch.no_grad():  # no gradients needed
    lo, lc = forward(X, W1_a, b1_a, W2_a, b2_a, Wo_a, bo_a, Wc_a, bc_a)  # forward pass on all 100
    ones_acc  = accuracy(lo, Y_ones)   # # of ones-digit correct
    carry_acc = accuracy(lc, Y_carry)  # carry accuracy (will be random since carry head was not trained)

print(f'Model A  ones accuracy : {ones_acc}/100')
print(f'Model A  carry accuracy: {carry_acc}/100  (will be random since carry head was not trained)')
print()
for i in range(100):                       # print predictions 
    a, b = i//10, i%10                     # original digits a and b from the index
    print(f'  {a} + {b}  ->  ones={lo.argmax(dim=1)[i].item()}  carry={lc.argmax(dim=1)[i].item()}')

Model A  ones accuracy : 100/100
Model A  carry accuracy: 46/100  (will be random since carry head was not trained)

  0 + 0  ->  ones=0  carry=1
  0 + 1  ->  ones=1  carry=1
  0 + 2  ->  ones=2  carry=1
  0 + 3  ->  ones=3  carry=1
  0 + 4  ->  ones=4  carry=1
  0 + 5  ->  ones=5  carry=1
  0 + 6  ->  ones=6  carry=1
  0 + 7  ->  ones=7  carry=1
  0 + 8  ->  ones=8  carry=1
  0 + 9  ->  ones=9  carry=1
  1 + 0  ->  ones=1  carry=1
  1 + 1  ->  ones=2  carry=1
  1 + 2  ->  ones=3  carry=1
  1 + 3  ->  ones=4  carry=1
  1 + 4  ->  ones=5  carry=1
  1 + 5  ->  ones=6  carry=1
  1 + 6  ->  ones=7  carry=1
  1 + 7  ->  ones=8  carry=1
  1 + 8  ->  ones=9  carry=1
  1 + 9  ->  ones=0  carry=1
  2 + 0  ->  ones=2  carry=1
  2 + 1  ->  ones=3  carry=1
  2 + 2  ->  ones=4  carry=1
  2 + 3  ->  ones=5  carry=1
  2 + 4  ->  ones=6  carry=1
  2 + 5  ->  ones=7  carry=1
  2 + 6  ->  ones=8  carry=1
  2 + 7  ->  ones=9  carry=1
  2 + 8  ->  ones=0  carry=1
  2 + 9  ->  ones=1  carry=1
  3 + 0  ->  

## Freeze Model A

In [100]:
for p in params_a:
    p.requires_grad_(False)  # turn off gradient tracking so Model A weights cannot be modified
print('Model A frozen')

Model A frozen


## Model B — both heads from scratch

In [101]:
# Same as Model A but now all parameters are included in the training and receive gradients, so both heads will learn together 
# and we expect to see both ones and carry accuracy improve over time until they both reach 100%
W1_b, b1_b, W2_b, b2_b, Wo_b, bo_b, Wc_b, bc_b = make_weights()                  
params_b = [W1_b, b1_b, W2_b, b2_b, Wo_b, bo_b, Wc_b, bc_b]                      

for p in params_b:
    p.requires_grad_(True) 
t0_b = time.time()  

for epoch in range(1, 50001):  

    logits_ones, logits_carry = forward(X, W1_b, b1_b, W2_b, b2_b, Wo_b, bo_b, Wc_b, bc_b)  

    loss = cross_entropy(logits_ones, Y_ones) + cross_entropy(logits_carry, Y_carry)  

    loss.backward()  

    with torch.no_grad():
        for p in params_b:
            p -= lr * p.grad   
            p.grad.zero_()     

    correct_ones  = accuracy(logits_ones,  Y_ones)   
    correct_carry = accuracy(logits_carry, Y_carry)  

    if epoch % 1000 == 0:
        print(f'epoch {epoch:5d}  loss={loss.item():.4f}  ones={correct_ones}/100  carry={correct_carry}/100')

    if correct_ones == 100 and correct_carry == 100:  
        print(f'Model B reached 100% on both at epoch {epoch}')
        break

time_b = time.time() - t0_b 
print(f'Model B training time: {time_b:.2f}s')

epoch  1000  loss=2.8076  ones=14/100  carry=93/100
epoch  2000  loss=2.3801  ones=29/100  carry=100/100
epoch  3000  loss=2.2386  ones=43/100  carry=100/100
epoch  4000  loss=2.1133  ones=53/100  carry=100/100
epoch  5000  loss=1.9325  ones=58/100  carry=100/100
epoch  6000  loss=1.6964  ones=67/100  carry=100/100
epoch  7000  loss=1.4216  ones=78/100  carry=100/100
epoch  8000  loss=1.0894  ones=88/100  carry=100/100
epoch  9000  loss=0.7287  ones=97/100  carry=100/100
Model B reached 100% on both at epoch 9663
Model B training time: 13.25s


## Verify Model B

In [102]:
with torch.no_grad():  
    lo, lc = forward(X, W1_b, b1_b, W2_b, b2_b, Wo_b, bo_b, Wc_b, bc_b)
    ones_acc  = accuracy(lo, Y_ones)
    carry_acc = accuracy(lc, Y_carry)

print(f'Model B  ones accuracy : {ones_acc}/100')
print(f'Model B  carry accuracy: {carry_acc}/100')
print()
for i in range(100):
    a, b = i//10, i%10
    print(f'  {a} + {b}  ->  ones={lo.argmax(dim=1)[i].item()}  carry={lc.argmax(dim=1)[i].item()}')

Model B  ones accuracy : 99/100
Model B  carry accuracy: 100/100

  0 + 0  ->  ones=0  carry=0
  0 + 1  ->  ones=1  carry=0
  0 + 2  ->  ones=2  carry=0
  0 + 3  ->  ones=3  carry=0
  0 + 4  ->  ones=4  carry=0
  0 + 5  ->  ones=5  carry=0
  0 + 6  ->  ones=7  carry=0
  0 + 7  ->  ones=7  carry=0
  0 + 8  ->  ones=8  carry=0
  0 + 9  ->  ones=9  carry=0
  1 + 0  ->  ones=1  carry=0
  1 + 1  ->  ones=2  carry=0
  1 + 2  ->  ones=3  carry=0
  1 + 3  ->  ones=4  carry=0
  1 + 4  ->  ones=5  carry=0
  1 + 5  ->  ones=6  carry=0
  1 + 6  ->  ones=7  carry=0
  1 + 7  ->  ones=8  carry=0
  1 + 8  ->  ones=9  carry=0
  1 + 9  ->  ones=0  carry=1
  2 + 0  ->  ones=2  carry=0
  2 + 1  ->  ones=3  carry=0
  2 + 2  ->  ones=4  carry=0
  2 + 3  ->  ones=5  carry=0
  2 + 4  ->  ones=6  carry=0
  2 + 5  ->  ones=7  carry=0
  2 + 6  ->  ones=8  carry=0
  2 + 7  ->  ones=9  carry=0
  2 + 8  ->  ones=0  carry=1
  2 + 9  ->  ones=1  carry=1
  3 + 0  ->  ones=3  carry=0
  3 + 1  ->  ones=4  carry=0
  3 + 

## Freeze Model B

In [103]:
for p in params_b:
    p.requires_grad_(False) 
print('Model B frozen')

Model B frozen


## Model C — fine-tune from Model A with first layer frozen

Model C starts with Model A's learned representation. The first hidden layer is frozen to preserve it. Only W2, b2, Wo, bo, Wc, bc are allowed to train.

In [104]:
# copy parameters from Model A and detach so they have no gradient history
W1_c = W1_a.detach().clone()  
b1_c = b1_a.detach().clone()  
W2_c = W2_a.detach().clone()  
b2_c = b2_a.detach().clone()  
Wo_c = Wo_a.detach().clone()  
bo_c = bo_a.detach().clone()  
Wc_c = Wc_a.detach().clone()  
bc_c = bc_a.detach().clone()  

# freeze W1 and b1 so the first layer is fixed and cannot be modified by training; 
W1_c.requires_grad_(False)  
b1_c.requires_grad_(False)  

# allow the remaining layers to train
W2_c.requires_grad_(True)   
b2_c.requires_grad_(True)   
Wo_c.requires_grad_(True)   
bo_c.requires_grad_(True)   
Wc_c.requires_grad_(True)   
bc_c.requires_grad_(True)  

params_c_trainable = [W2_c, b2_c, Wo_c, bo_c, Wc_c, bc_c]  # only include trainable params in the update list

print('Model C: W1 frozen, all other params trainable')

Model C: W1 frozen, all other params trainable


In [105]:
t0_c = time.time()  # start timer for Model C

for epoch in range(1, 50001):  

    logits_ones, logits_carry = forward(X, W1_c, b1_c, W2_c, b2_c, Wo_c, bo_c, Wc_c, bc_c)  # forward pass

    loss = cross_entropy(logits_ones, Y_ones) + cross_entropy(logits_carry, Y_carry)  # train both heads

    loss.backward()  # backward pass W1_c and b1_c are skipped

    with torch.no_grad():
        for p in params_c_trainable:  # iterate over the unfrozen parameters
            p -= lr * p.grad          # gradient descent step
            p.grad.zero_()            # reset gradient for the next iteration

    correct_ones  = accuracy(logits_ones,  Y_ones)
    correct_carry = accuracy(logits_carry, Y_carry)

    if epoch % 1000 == 0:
        print(f'epoch {epoch:5d}  loss={loss.item():.4f}  ones={correct_ones}/100  carry={correct_carry}/100')

    if correct_ones == 100 and correct_carry == 100:  # early stop when both heads are perfect
        print(f'Model C reached 100% on both at epoch {epoch}')
        break

time_c = time.time() - t0_c  # record Model C training duration
print(f'Model C training time: {time_c:.2f}s')

epoch  1000  loss=0.8215  ones=100/100  carry=87/100
epoch  2000  loss=0.4984  ones=100/100  carry=99/100
epoch  3000  loss=0.3298  ones=100/100  carry=99/100
Model C reached 100% on both at epoch 3065
Model C training time: 3.72s


## Verify Model C

In [106]:
with torch.no_grad():
    lo, lc = forward(X, W1_c, b1_c, W2_c, b2_c, Wo_c, bo_c, Wc_c, bc_c)
    ones_acc  = accuracy(lo, Y_ones)
    carry_acc = accuracy(lc, Y_carry)

print(f'Model C  ones accuracy : {ones_acc}/100')
print(f'Model C  carry accuracy: {carry_acc}/100')
print()
for i in range(100):
    a, b = i//10, i%10
    print(f'  {a} + {b}  ->  ones={lo.argmax(dim=1)[i].item()}  carry={lc.argmax(dim=1)[i].item()}')

Model C  ones accuracy : 100/100
Model C  carry accuracy: 100/100

  0 + 0  ->  ones=0  carry=0
  0 + 1  ->  ones=1  carry=0
  0 + 2  ->  ones=2  carry=0
  0 + 3  ->  ones=3  carry=0
  0 + 4  ->  ones=4  carry=0
  0 + 5  ->  ones=5  carry=0
  0 + 6  ->  ones=6  carry=0
  0 + 7  ->  ones=7  carry=0
  0 + 8  ->  ones=8  carry=0
  0 + 9  ->  ones=9  carry=0
  1 + 0  ->  ones=1  carry=0
  1 + 1  ->  ones=2  carry=0
  1 + 2  ->  ones=3  carry=0
  1 + 3  ->  ones=4  carry=0
  1 + 4  ->  ones=5  carry=0
  1 + 5  ->  ones=6  carry=0
  1 + 6  ->  ones=7  carry=0
  1 + 7  ->  ones=8  carry=0
  1 + 8  ->  ones=9  carry=0
  1 + 9  ->  ones=0  carry=1
  2 + 0  ->  ones=2  carry=0
  2 + 1  ->  ones=3  carry=0
  2 + 2  ->  ones=4  carry=0
  2 + 3  ->  ones=5  carry=0
  2 + 4  ->  ones=6  carry=0
  2 + 5  ->  ones=7  carry=0
  2 + 6  ->  ones=8  carry=0
  2 + 7  ->  ones=9  carry=0
  2 + 8  ->  ones=0  carry=1
  2 + 9  ->  ones=1  carry=1
  3 + 0  ->  ones=3  carry=0
  3 + 1  ->  ones=4  carry=0
  3 +

## Compare training times

In [107]:
print('='*40)
print(f'Model A (ones only, from scratch) : {time_a:.2f}s')  # ones head only
print(f'Model B (both heads, from scratch) : {time_b:.2f}s') # both heads cold start
print(f'Model C (both heads, frozen W1)    : {time_c:.2f}s') # both heads warm start from A
print('='*40)
if time_c < time_b:  # check which model converged faster
    print(f'C was faster than B by {time_b - time_c:.2f}s')
else:
    print(f'B was faster than C by {time_c - time_b:.2f}s')

if time_c+time_a < time_b:  # check which model converged faster
    print(f'A and C were faster than B by {time_b - (time_c+time_a):.2f}s')
else:
    print(f'B was faster than A and C by {time_c+time_a - time_b:.2f}s')

Model A (ones only, from scratch) : 10.22s
Model B (both heads, from scratch) : 13.25s
Model C (both heads, frozen W1)    : 3.72s
C was faster than B by 9.53s
B was faster than A and C by 0.69s
